# Dijital İkiz İnsan — Kaggle Eğitim Notebook'u

Bu notebook, TÜBİTAK projesinin gerçek ML aşaması için hazırlanmıştır. Eğitim Kaggle GPU üzerinde yapılır; bu bilgisayarın GPU'suna ihtiyaç duymaz.

**Görevler:** VAD (`valence`, `arousal`, `dominance`) ve Big Five (`openness`, `conscientiousness`, `extraversion`, `agreeableness`, `neuroticism`) regresyonu.

**Önemli:** Gerçek sonuçlar yalnızca gerçek, izin/lisansı belgelenmiş veriyle eğitim ve test çalıştırıldıktan sonra raporlanmalıdır. %75 hedefi önceden başarı olarak kabul edilmez.

## 0) Kaggle ayarları

Notebook sağ panelinde **Accelerator = GPU** seç. Model indirileceği için **Internet = On** gerekebilir. Kaggle ortamı uygun bir GPU sağlamazsa notebook CPU ile devam edebilir; eğitim ciddi biçimde yavaşlar.

In [ ]:
!nvidia-smi

import os, sys, json, math, random, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
   

In [ ]:
!pip install -q -U transformers accelerate safetensors scikit-learn pandas


## 1) Veri setini bul

Kaggle'ın sağ tarafındaki **Add Data** ile kendi etiketi/lisansı belgelenmiş CSV veri setini notebook'a ekle. Notebook, `/kaggle/input` altında uygun CSV'yi otomatik arar.

Beklenen temel kolonlar: `participant_id`, `text` ve eğitim yapılacak hedef kolonlar. Araştırma verisinde `source` ile `license_or_consent` bilgileri de tutulmalıdır.

In [ ]:
required_any = {'participant_id', 'text'}
csv_files = sorted(Path('/kaggle/input').rglob('*.csv'))
if not csv_files:
    raise FileNotFoundError('Kaggle Input altında CSV bulunamadı. Sağ panelden Add Data ile veri setini ekle.')

candidate = None
for path in csv_files:
    try:
        columns = set(pd.read_csv(path, nrows=5).columns)
    except Exception:
        continue
    if required_any.issubset(columns):

           candidate = path
        break

DATA_PATH = candidate
print('Seçilen veri:', DATA_PATH)
df = pd.read_csv(DATA_PATH)
print('Satır:', len(df))
print('Kolonlar:', list(df.columns))
display(df.head())

In [ ]:
TARGETS = {
    'vad': ['valence', 'arousal', 'dominance'],
    'big_five': ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism'],
}

def validate_frame(frame, labels):
    required = {'participant_id', 'text', *labels}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f'Eksik kolonlar: {missing}')
    clean = frame.dropna(subset=['participant_id', 'text', *labels]).copy()
    clean['participant_id'] = clean['participant_id'].astype(str).str.strip()
    clean['text'] = clean['text'].astype(str).str.strip()
    for label in labels:
        clean[label] = pd.to_numeric(clean[label], errors='coerce')
    clean = clean.dropna(subset=labels)
    if (clean[labels] < 0).any().any() or (clean[labels] > 1).any().any():
        raise ValueError('Etiketler 0–1 aralığında olmalı; önce normalize et.')
    if clean['participant_id'].eq('').any() or clean['text'].eq('').any():
        raise ValueError('participant_id ve text boş bırakılamaz.')
    return clean

vad_df = validate_frame(df, TARGETS['vad'])
big5_df = validate_frame(df, TARGETS['big_five'])
print('VAD örnekleri:', len(vad_df), 'katılımcı:', vad_df.participant_id.nunique())
print('Big Five örnekleri:', len(big5_df), 'katılımcı:', big5_df.participant_id.nunique())

## 2) Katılımcı bazlı train/validation/test bölmesi

Aynı katılımcının örneklerini farklı bölümlere dağıtmıyoruz. Bu, kişi bilgisinin test setine sızmasını önlemek için kritik.

In [ ]:
def split_by_participant(frame, train_ratio=0.8, val_ratio=0.1, seed='digital-twin-v1'):
    participants = sorted(frame['participant_id'].unique())
    assignments = {}
    for participant in participants:
        digest = hashlib.sha256(f'{seed}:{participant}'.encode('utf-8')).hexdigest()
        value = int(digest[:8], 16) / 0xFFFFFFFF
        if value < train_ratio:
            assignments[participant] = 'train'
        elif value < train_ratio + val_ratio:
            assignments[participant] = 'validation'
        else:
            assignments[participant] = 'test'
    out = {name: frame[frame['participant_id'].map(assignments.__getitem__) == name].copy()
           for name in ('train', 'validation', 'test')}
    print({k: {'rows': len(v), 'participants': v.participant_id.nunique()} for k,v in out.items()})
    return out

vad_split = split_by_participant(vad_df)
big5_split = split_by_participant(big5_df)

## 3) Türkçe BERT regresyon modeli

Temel encoder: `dbmdz/bert-base-turkish-cased`. Kayıp fonksiyonu MSE, raporlanan test ölçümleri MAE ve RMSE'dir.

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MODEL_NAME = 'dbmdz/bert-base-turkish-cased'
MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
SEED = 42

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

class TextRegressionDataset(Dataset):
    def __init__(self, frame, tokenizer, labels):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.labels = labels
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        enc = self.tokenizer(str(row['text']), truncation=True, padding='max_length', max_length=MAX_LENGTH, return_tensors='pt')
        y = torch.tensor([float(row[x]) for x in self.labels], dtype=torch.float32)
        return {'input_ids': enc['input_ids'].squeeze(0), 'attention_mask': enc['attention_mask'].squeeze(0), 'labels': y}

class TransformerRegressor(nn.Module):
    def __init__(self, model_name, output_dim, dropout=0.1):
        super().__init__(); self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout); self.head = nn.Linear(self.encoder.config.hidden_size, output_dim)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.head(self.dropout(out.last_hidden_state[:, 0]))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Cihaz:', device)


In [ ]:
def evaluate(model, loader, device):
    model.eval(); preds=[]; ys=[]
    with torch.no_grad():
        for batch in loader:
            pred = model(batch['input_ids'].to(device), batch['attention_mask'].to(device)).cpu().numpy()
            preds.append(pred); ys.append(batch['labels'].numpy())
    y_pred = np.clip(np.concatenate(preds), 0, 1)
    y_true = np.concatenate(ys)
    per_label = {}
    for i, label in enumerate(CURRENT_LABELS):
        per_label[label] = {
            'mae': float(mean_absolute_error(y_true[:, i], y_pred[:, i])),
            'rmse': float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))),
            'r2': float(r2_score(y_true[:, i], y_pred[:, i])) if len(y_true) > 1 else None,
        }
    return {'mae': float(mean_absolute_error(y_true, y_pred)), 'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))), 'per_label': per_label}

def train_task(split, labels, output_dir):
    global CURRENT_LABELS
    CURRENT_LABELS = labels
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_loader = DataLoader(TextRegressionDataset(split['train'], tokenizer, labels), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TextRegressionDataset(split['validation'], tokenizer, labels), batch_size=BATCH_SIZE)
    test_loader = DataLoader(TextRegressionDataset(split['test'], tokenizer, labels), batch_size=BATCH_SIZE)
    model = TransformerRegressor(MODEL_NAME, len(labels)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.MSELoss()
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    best = float('inf')
    for epoch in range(1, EPOCHS+1):
        model.train(); losses=[]
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            ids=batch['input_ids'].to(device); mask=batch['attention_mask'].to(device); y=batch['labels'].to(device)
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                pred=model(ids, mask); loss=loss_fn(pred, y)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); losses.append(float(loss.detach().cpu()))
        val = evaluate(model, val_loader, device)
        print(f'epoch={epoch} train_mse={np.mean(losses):.5f} val_mae={val["mae"]:.5f} val_rmse={val["rmse"]:.5f}')
        if val['mae'] < best:
            best = val['mae']; torch.save(model.state_dict(), output_dir/'best_model.pt')
    model.load_state_dict(torch.load(output_dir/'best_model.pt', map_location=device))
    test = evaluate(model, test_loader, device)
    tokenizer.save_pretrained(output_dir/'tokenizer')
    report={'model':MODEL_NAME,'labels':labels,'device':str(device),'seed':SEED,'epochs':EPOCHS,'batch_size':BATCH_SIZE,'learning_rate':LEARNING_RATE,'max_length':MAX_LENGTH,'train_rows':len(split['train']),'validation_rows':len(split['validation']),'test_rows':len(split['test']),'test_metrics':test,'scientific_note':'Gerçek test çalıştırılmadan rapora sonuç yazılmamalıdır.'}
    (output_dir/'metrics.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    return report


## 4) Eğitim

Önce VAD, sonra Big Five eğit. İkisi ayrı çıktı klasörlerinde tutulur.

In [ ]:
vad_report = train_task(vad_split, TARGETS['vad'], '/kaggle/working/artifacts/vad')
print(json.dumps(vad_report['test_metrics'], ensure_ascii=False, indent=2))

In [ ]:
big5_report = train_task(big5_split, TARGETS['big_five'], '/kaggle/working/artifacts/big-five')
print(json.dumps(big5_report['test_metrics'], ensure_ascii=False, indent=2))

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/dijital-ikiz-model-artifacts', 'zip', '/kaggle/working/artifacts')
print('Artifact paketi:', archive)
print('Metrikler:')
for p in Path('/kaggle/working/artifacts').glob('*/metrics.json'):
    print('\n', p)
    print(p.read_text(encoding='utf-8'))

## 5) Sonuçların yorumu

Buradaki `metrics.json` dosyalarını deney günlüğüne aktar. Test MAE/RMSE/R² değerlerini gerçekten çalıştırılmış sonuç olarak kaydet.

Model araştırmada yeterli bulunursa sonraki aşama: modeli uygun biçimde paketlemek ve uygulamanın sunucu tarafına entegre etmek. Büyük model ağırlıklarını doğrudan Vercel istemci koduna koyma.